# ShercliffDuct3D — 3D coupled MHD duct (two-way)
`MagneticCoupled3D` · validation form (TRUST-LiMe).


## Introduction
3D counterpart of the 2D Shercliff duct: coupled low-Rm MHD in a duct
with insulating walls, driven by an imposed pressure difference, with
`force_lorentz` on momentum and `vitesse_hydraulique` + `induction_stretching`
on the magnetic equation. Smoke-level form: the run must complete and the
probes must stay finite.


In [ ]:
from trustutils import run
run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read TRUST probe (`.son`) file
Case copied by `trustutils` into `run.BUILD_DIRECTORY`.


In [ ]:
import numpy as np
import os
from trustutils.files import SonFile
BUILD = run.BUILD_DIRECTORY
def read_son(case, name, sub=None):
    if sub is not None:
        cands = [f"{BUILD}/{sub}/{case}_{name}.son",
                 f"{BUILD}/{sub}/PAR_{case}_{name}.son"]
    else:
        cands = [f"{BUILD}/coarse/{case}_{name}.son",
                 f"{BUILD}/{case}_{name}.son"]
    for p in cands:
        if os.path.exists(p):
            s = SonFile(p, None)
            entries = s.getEntries()
            assert entries, f"no probe entry found in {p}"
            t, v = s.getValues(entries[0])
            return np.asarray(t, dtype=float), np.asarray(v, dtype=float)
    raise FileNotFoundError(cands)



## Run
Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant.


In [ ]:
NB_PROCS = 2
run.reset()
NX0, NY0, NZ0 = 5, 9, 5
meshes = {
    "coarse": {"nx": NX0, "ny": NY0, "nz": NZ0, "tmax": 2, "pmag": 0.5, "eta": 0.05},
}
cases = {}
for lvl, dic in meshes.items():
    cases[lvl] = run.addCaseFromTemplate(
        "shercliff_duct_3d.data", lvl, dic=dic,
        nbProcs=NB_PROCS, excluNR=True)
    cases[lvl].partition()
run.printCases()
run.runCases()
case = cases["coarse"]



## Analytic comparison status

The standalone `Shercliff` form (MagneticStandalone) carries full analytic
comparisons against the insulating-wall Shercliff profile. This **coupled**
case (Tassone et al. Fig. 6, conducting walls, Lorentz feedback) has a
different analytic solution (Shercliff series for conducting walls) that is
**not yet implemented** here — only finiteness and sign sanity are checked.
Adding the Tassone analytic requires the Shercliff series coefficients for
the given (Ha, Re, conductance ratio); this is an open item.


## Checks
Smoke-level: the run completed and the probes are finite.


In [ ]:
# Sanity checks on the Tassone (Shercliff) coupled MHD duct.
# Full analytic comparison (Shercliff series for conducting walls) is an open
# item: the standalone Shercliff analytic (insulating, no Lorentz force) does
# NOT apply to this coupled conducting-wall case.
import numpy as np

# Velocity probes (sonde_a1..a6): must be finite and non-negative (flow in +x).
vel_ok = True
for k in range(1, 7):
    try:
        t, v = read_son(case.dataFileName_, f"SONDE_A{k}")
        v = np.asarray(v)
        assert np.all(np.isfinite(v)), f"SONDE_A{k} non-finite"
        assert np.all(v >= -1e-12), f"SONDE_A{k} has negative velocity: min={v.min():.3e}"
        print(f"SONDE_A{k}: last={float(v[-1]):.6g}  (finite, >=0)")
    except Exception as e:
        print(f"SONDE_A{k} read skipped: {e}")

# Magnetic-field probes: finite.
for name in ("SONDE_BX", "SONDE_BZ"):
    try:
        t, b = read_son(case.dataFileName_, name)
        b = np.asarray(b)
        assert np.all(np.isfinite(b)), f"{name} non-finite"
        print(f"{name}: last={float(b[-1]):.6g}  (finite)")
    except Exception as e:
        print(f"{name} read skipped: {e}")

print("OK: ShercliffDuct3D run completed (finiteness + sign sanity).")
print("NOTE: full Tassone analytic (Shercliff series, conducting walls) not yet asserted.")


## Computer Performance


## References

- Shercliff, J.A. (1953) *Steady motion of conducting fluids in pipes under
  transverse magnetic fields*, Proc. Camb. Phil. Soc. 49, 136.
- Shercliff, J.A. (1962) *The theory of electromagnetic flow-measurement*,
  Cambridge University Press. (Full derivation of the Shercliff series.)
- Tassone, A. et al. (2024) *MHD in breeding blankets*, Fig. 6 —
  dimensionless parameter ranges and comparison data.
- arXiv:2603.02853 (2026) *Two-phase stratified MHD flows in rectangular
  ducts* — series forms for two-phase flows (Appendix A/B). **Not directly
  applicable to single-phase** (degenerate at h=1); the single-phase
  Shercliff series requires Shercliff 1962 ch. 4.
- COMSOL V&V note (semanticscholar bd35/9d74) — Shercliff/Hunt
  benchmark setup and thin-wall conductance ratio.

**Open item:** the full Tassone analytic (Shercliff series, single-phase
conducting walls) is not yet asserted. The arXiv two-phase series does not
directly apply (degenerate single-phase limit); the single-phase series
must be derived from Shercliff 1962 ch. 4. See plan `mhd_coupled.md` 2.4.


In [ ]:
# Insulating-wall Shercliff analytic (standalone reference limit).
# The magnetic walls are insulating (Neumann) here, so the standalone
# Shercliff profile is a useful reference in the weak-coupling / low-Rm limit.
# The coupled case is NOT expected to match exactly (Lorentz force modifies
# the velocity); this is a diagnostic, not a hard assert.
H, ETA, MU0 = 0.5, 0.05, 1.0
def b_x_steady_insulating(y, U0=1.0, B0=1.0):
    return (B0 * U0 / ETA) * (y**3 / (3.0 * H**2) - y)
def j_z_steady_insulating(y, U0=1.0, B0=1.0):
    return (B0 * U0 / (ETA * MU0)) * (1.0 - (y / H) ** 2)

# Probe locations from the data file (x=0.375, z=0, y varies).
probe_ys = {"SONDE_A1": 0.0, "SONDE_A2": 0.0, "SONDE_A3": 0.0,
            "SONDE_A4": 0.25, "SONDE_A5": 0.45, "SONDE_A6": 0.25}
print("Insulating-wall analytic reference (U0=1, B0=1, low-Rm limit):")
for name, y in probe_ys.items():
    try:
        t, v = read_son(case.dataFileName_, name)
        v = np.asarray(v)
        print(f"  {name}: coupled u={float(v[-1]):.6g}  (insulating parabolic at y={y}: {1.0*(1.0-(y/H)**2):.6g})")
    except Exception as e:
        print(f"  {name} read skipped: {e}")
print("NOTE: differences are expected (Lorentz force modifies the velocity).")


In [ ]:
run.tablePerf()
